# VIIRS vegetation-predictor ET downscaling — cell-by-cell test

SEBAL O'ZGARMAYDI. VIIRS = faqat VNP09GA I1/I2/I3 predictor.
Target: `lambda` (EVAP_FRAC) yoki `kc` (KC).

In [1]:
# Cell 1: ee.Initialize va CONFIG
import ee
ee.Initialize(project='carbon-science-461016-q2')
from sebal_gee_v4 import ee_utils; ee_utils.install_getinfo_retry()
from sebal_gee_v4 import main as M
from sebal_gee_v4 import viirs_downscaling as vds

MODE  = 'lambda'   # 'lambda' yoki 'kc'
MODEL = 'ndvi'     # 'ndvi' | 'ndvi2' | 'multi'
QA    = 'lenient'
START, END = '2026-03-01', '2026-03-31'
print('config OK')

config OK


In [2]:
# Cell 2: ROI va month
roi = (ee.FeatureCollection('FAO/GAUL/2015/level1')
       .filter(ee.Filter.eq('ADM1_NAME', 'Sirdarya')).geometry())
print('ROI OK')

ROI OK


In [3]:
# Cell 3: Landsat/HLS SEBAL anchor output
scenes, info = M.process_tile(roi, START, END, 'pysebal', 'BOTH', 70, '')
anchors = [{'image': ee.Image(s), 'date': info['dates'][i]}
           for i, s in enumerate(scenes)]
print('Anchorlar:', len(anchors), '| sanalar:', info['dates'])
print('Bandlar:', anchors[0]['image'].bandNames().getInfo())

    ⚠️  Cloud precheck FAIL: 87.3% (cropland, limit=30%) → skip  [1_LC08_154032_20260306]
    ⚠️  Cloud precheck FAIL: 74.4% (cropland, limit=30%) → skip  [1_LC08_154032_20260322]
    ✅ Cloud precheck OK: 0.4% (cropland, limit=30%)  [1_LC08_155032_20260313]
    ✅ Cloud precheck OK: 6.7% (cropland, limit=30%)  [2_LC09_155032_20260321]
   Tasvirlar: 2 | ['2026-03-13', '2026-03-21']
   Sahna 1/2...
  🌾 Anchor cropland mask pixel count: {'SLOPE': 384477}
   Sahna 2/2...
  🌾 Anchor cropland mask pixel count: {'SLOPE': 384477}
Anchorlar: 2 | sanalar: ['2026-03-13', '2026-03-21']
Bandlar: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'WATER_MASK', 'LST', 'DEM', 'SLOPE', 'WIND_SPEED_10M', 'AIR_TEMP', 'PRESSURE', 'SSRD', 'STRD', 'DEWPOINT', 'RHO_AIR', 'NDVI', 'SAVI', 'ALBEDO', 'EMISSIVITY', 'Z0M', 'Z0H', 'TAU_SW', 'LAI', 'K_DOWN', 'L_DOWN'

In [4]:
# Cell 4: VNP09GA I1/I2/I3 va predictorlar
vimg = vds.get_viirs_vnp09ga(anchors[0]['date'], roi)
vmask = vds.mask_viirs_vnp09ga(vimg, QA)
preds = vds.compute_viirs_predictors(vmask)
print('Predictor bandlar:', preds.bandNames().getInfo())
print('NDVI stats:', preds.select('NDVI').reduceRegion(
    ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    roi, 500, maxPixels=1e9, bestEffort=True).getInfo())

Predictor bandlar: ['NDVI', 'EVI2', 'NDWI']
NDVI stats: {'NDVI_max': None, 'NDVI_mean': None, 'NDVI_min': None}


In [5]:
# Cell 5: target (Lambda/KC) ni VIIRS gridga aggregate
vproj = vds.get_viirs_projection(vimg)
print('VIIRS scale (m):', vproj.nominalScale().getInfo())
tband = 'EVAP_FRAC' if MODE == 'lambda' else 'KC'
tcoarse = vds.aggregate_to_viirs_grid(anchors[0]['image'].select(tband), vproj)
print('target_coarse mean:', tcoarse.reduceRegion(
    ee.Reducer.mean(), roi, vproj.nominalScale(),
    maxPixels=1e9, bestEffort=True).getInfo())

VIIRS scale (m): 463.3127165279165


EEException: User memory limit exceeded.

In [ ]:
# Cell 6: regression fit
fc = vds.build_training_samples(anchors, roi, vproj, MODE, QA)
snp = vds.samples_to_numpy(fc)
reg = vds.fit_viirs_regression(snp, MODEL)
print('Regressiya:', reg)

In [ ]:
# Cell 7: weight qurish va mean(W) ≈ 1 tekshirish
weight = vds.build_spatial_weight(anchors[0]['image'].select(tband), vproj, MODE)
wcoarse = vds.aggregate_to_viirs_grid(weight, vproj)
print('mean(W) over VIIRS (≈1):', wcoarse.reduceRegion(
    ee.Reducer.mean(), roi, vproj.nominalScale(),
    maxPixels=1e9, bestEffort=True).getInfo())

In [ ]:
# Cell 8: bitta VIIRS clear kunda coarse prediction
TEST_DAY = '2026-03-15'
vt = vds.mask_viirs_vnp09ga(vds.get_viirs_vnp09ga(TEST_DAY, roi), QA)
pt = vds.compute_viirs_predictors(vt).reproject(crs=vproj)
coarse_pred = vds.predict_coarse_target(pt, reg, MODEL, MODE)
print('coarse target bashorat:', coarse_pred.reduceRegion(
    ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    roi, vproj.nominalScale(), maxPixels=1e9, bestEffort=True).getInfo())

In [ ]:
# Cell 9: 30 m downscaling
t30 = vds.downscale_target_to_30m(coarse_pred, weight, vproj)
print('target_30 stats:', t30.reduceRegion(
    ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    roi, 100, maxPixels=1e9, bestEffort=True).getInfo())

In [ ]:
# Cell 10: conservation error
print('conservation:', vds.compute_conservation_metrics(
    t30, coarse_pred, roi, vproj).getInfo())

In [ ]:
# Cell 11: daily ET
if MODE == 'lambda':
    alb, tau = vds.interp_radiation_bands(anchors, TEST_DAY)
    rn24 = vds.daily_rn24(TEST_DAY, roi, alb, tau)
    et = vds.compute_daily_et_lambda_mode(t30, rn24)
else:
    etref = vds._daily_etref(anchors, TEST_DAY, roi)
    et = vds.compute_daily_et_kc_mode(t30, etref)
print('ET_24 stats:', et.reduceRegion(
    ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    roi, 100, maxPixels=1e9, bestEffort=True).getInfo())

In [ ]:
# Cell 12: oylik ET sum (HAR KUN, faqat mavjudlar emas)
daily = vds.build_daily_viirs_downscaled_collection(
    START, END, roi, anchors, reg, MODEL, MODE, vproj, QA, 'linear')
monthly = vds.build_monthly_et_sum(daily)
print('Kunlar:', len(daily))
print('Oylik ET (mm/oy):', monthly.reduceRegion(
    ee.Reducer.mean(), roi, 200, maxPixels=1e9, bestEffort=True).getInfo())

In [ ]:
# Cell 13: hold-out validation
if len(anchors) >= 2:
    for mdl in (['ndvi', 'ndvi2', 'multi'] if True else [MODEL]):
        res = vds.validate_holdout(anchors, 0, roi, vproj, mdl, MODE, QA)
        print(mdl, '→', res)
else:
    print('Hold-out uchun 2+ anchor kerak — diapazonni kengaytiring.')

In [ ]:
0: T42TVK_20260306T060501
1: T42TVK_20260313T061106
2: T42TVK_20260314T060506
3: T42TVK_20260321T061112
4: T42TVK_20260322T060452
5: T42TVK_20260329T061057
6: T42TVK_20260330T060501

0: T42TVK_20260301T060659
1: T42TVK_20260306T060721
2: T42TVK_20260309T061711
3: T42TVK_20260311T060629
4: T42TVK_20260314T061629
5: T42TVK_20260316T060631
6: T42TVK_20260321T060629
7: T42TVK_20260324T061629
8: T42TVK_20260328T061311
9: T42TVK_20260329T061631
10: T42TVK_20260331T062321